# Step 2: Semantic Retrieval

Step 1 (exact + fuzzy matching) resolves the literals that look similar to something we have already seen in training. But what about literals that use completely different wording? For those, we need a method that compares the medical *meaning* of the literal against the catalog of all possible ICD codes.

**The idea:** treat the 180K official ICD descriptions as a search index. For each unresolved test literal, find the description that is most similar to it, and predict that description's code.

**The technique:** TF-IDF vectorization with two complementary analyzers:
- Character n-grams (3-5 chars): catches typos, abbreviations, partial word matches
- Word n-grams (1-2 words): catches exact word and phrase matches

We combine the scores from both. Similarity is measured by cosine similarity.

Files to upload before running:
- `train_preprocessed.csv` (from Step 0)
- `test_preprocessed.csv` (from Step 0)
- `reference_preprocessed.csv` (from Step 0)
- `step1_predictions.csv` (from Step 1)

Files produced at the end:
- `step2_predictions.csv` (test predictions combining Steps 1 and 2)

## 1. Imports and loading

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('train_preprocessed.csv')
test = pd.read_csv('test_preprocessed.csv')
reference = pd.read_csv('reference_preprocessed.csv')
step1 = pd.read_csv('step1_predictions.csv')

print(f'Training:           {len(train):,} rows')
print(f'Test:               {len(test):,} rows')
print(f'Reference (ICD-10): {len(reference):,} rows')
print(f'Step 1 predictions: {len(step1):,} rows')
print(f'\nStep 1 method breakdown:')
print(step1['step1_method'].value_counts())

Training:           13,700 rows
Test:               6,667 rows
Reference (ICD-10): 179,742 rows
Step 1 predictions: 6,667 rows

Step 1 method breakdown:
step1_method
exact         3812
fuzzy         2819
unresolved      36
Name: count, dtype: int64


## 2. Look at the reference data we will search through

Each row in the reference is a (Code, Description) pair. Some are diagnosis codes (D), some are procedure codes (P). The descriptions are in formal Spanish and can be quite long.

In [3]:
reference[['Code', 'D_P', 'Description', 'Description_norm']].sample(10, random_state=42)

,Code,D_P,Description,Description_norm
35,A038,D,Otras shigelosis,otras shigelosis
4739,E7200,D,"Trastornos del transporte de aminoácidos, no e...","trastornos del transporte de aminoacidos, no e..."
35054,S142XX,D,Traumatismo de raíz nerviosa de columna cervical,traumatismo de raiz nerviosa de columna cervical
94061,W04XXXS,D,Caída al ser llevado o cargado por otras perso...,caida al ser llevado o cargado por otras perso...
48687,S5343,D,Esguince del ligamento colateral radial,esguince del ligamento colateral radial
29356,Q422,D,"Agenesia congénita, atresia y estenosis del an...","agenesia congenita, atresia y estenosis del an..."
23710,M90679,D,Osteítis deformante en enfermedades neoplásica...,osteitis deformante en enfermedades neoplasica...
63066,S82236R,D,Fractura oblicua sin desplazamiento de diáfisi...,fractura oblicua sin desplazamiento de diafisi...
13452,J960,D,Insuficiencia respiratoria aguda,insuficiencia respiratoria aguda
12272,I70442,D,Aterosclerosis de injerto(s) de derivación ven...,aterosclerosis de injerto(s) de derivacion ven...


In [4]:
# Handle the rare case where Description_norm is missing (empty descriptions)
print(f'Empty Description_norm rows: {reference["Description_norm"].isna().sum()}')
reference['Description_norm'] = reference['Description_norm'].fillna('')
print(f'After fillna: {reference["Description_norm"].isna().sum()}')

Empty Description_norm rows: 0
After fillna: 0


## 3. Build the TF-IDF search indexes

We build two indexes:

**Character index:** uses overlapping character sequences of length 3 to 5. Example: "asma" with 3-grams becomes ['asm', 'sma']. This is the key trick that lets us match across typos and abbreviations because morphologically similar words share many character sequences.

**Word index:** uses single words and word pairs. This is better when the literal uses standard medical terminology that appears word-for-word in the reference.

Both indexes use sublinear TF (which dampens the effect of very common terms) and ignore very rare or very common n-grams to reduce noise.

This cell builds the indexes once and reuses them throughout the notebook. Building takes 30-60 seconds.

In [5]:
print('Building character n-gram index over 180K descriptions...')
start = time.time()

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',      # character n-grams within word boundaries
    ngram_range=(3, 5),       # 3 to 5 character sequences
    min_df=2,                 # ignore n-grams that appear in fewer than 2 descriptions
    max_df=0.95,              # ignore n-grams that appear in more than 95% of descriptions
    sublinear_tf=True         # use 1 + log(tf) instead of raw tf
)
char_ref_vectors = char_vectorizer.fit_transform(reference['Description_norm'])

print(f'  Shape: {char_ref_vectors.shape}')
print(f'  Built in {time.time()-start:.1f}s')
print()

print('Building word n-gram index over 180K descriptions...')
start = time.time()

word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),       # single words and word pairs
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
word_ref_vectors = word_vectorizer.fit_transform(reference['Description_norm'])

print(f'  Shape: {word_ref_vectors.shape}')
print(f'  Built in {time.time()-start:.1f}s')

Building character n-gram index over 180K descriptions...
  Shape: (179742, 38809)
  Built in 12.0s

Building word n-gram index over 180K descriptions...
  Shape: (179742, 29888)
  Built in 2.0s


## 4. Test retrieval on a few examples first

Before running on the whole test set, let's verify the indexes work by manually querying a few literals — including some that Step 1 could not resolve.

In [6]:
def search_one(query_text, top_k=5):
    """
    Find the top-k most similar reference descriptions for a query string.
    Returns a list of (code, description, char_score, word_score, combined_score).
    """
    # Vectorize the query with both indexes
    q_char = char_vectorizer.transform([query_text])
    q_word = word_vectorizer.transform([query_text])
    
    # Compute cosine similarity against all reference rows
    char_sims = cosine_similarity(q_char, char_ref_vectors).flatten()
    word_sims = cosine_similarity(q_word, word_ref_vectors).flatten()
    
    # Combine: simple average of the two scores
    combined = (char_sims + word_sims) / 2
    
    # Take the top-k highest-scoring indices
    top_idx = np.argsort(combined)[::-1][:top_k]
    
    results = []
    for i in top_idx:
        results.append({
            'code': reference.iloc[i]['Code'],
            'description': reference.iloc[i]['Description'],
            'char_score': char_sims[i],
            'word_score': word_sims[i],
            'combined': combined[i],
        })
    return results

# Try some examples that Step 1 couldn't resolve
demo_queries = [
    'shock septico',
    'temblor esencial familiar',
    'contusion rodilla izquierda',
    'neoplasia vesical infiltrante',
    'sindrome de abstinencia',
]

for q in demo_queries:
    print(f'Query: "{q}"')
    for r in search_one(q, top_k=3):
        print(f'  [{r["combined"]:.3f}] {r["code"]:10} {r["description"]}')
    print()

Query: "shock septico"
  [0.726] R6521      Sepsis grave con shock séptico
  [0.719] R6520      Sepsis grave sin shock séptico
  [0.694] T8112      Shock séptico después de un procedimiento

Query: "temblor esencial familiar"
  [0.862] G250       Temblor esencial
  [0.623] R251       Temblor no especificado
  [0.471] G251       Temblor inducido por fármacos

Query: "contusion rodilla izquierda"
  [0.883] S8002X     Contusión de rodilla izquierda
  [0.883] S8002      Contusión de rodilla izquierda
  [0.770] S8002XS    Contusión de rodilla izquierda, secuela

Query: "neoplasia vesical infiltrante"
  [0.623] R301       Tenesmo vesical
  [0.411] Q641       Extrofia vesical
  [0.367] N320       Obstrucción de cuello vesical

Query: "sindrome de abstinencia"
  [0.546] E2601      Síndrome de Conn
  [0.539] E7203      Síndrome de Lowe
  [0.534] G937       Síndrome de Reye



## 5. Run retrieval on the full test set

This is the slow part: we compute similarity between every test literal and all 180K reference descriptions. To stay efficient and memory-safe we work in small batches.

We also store the best score from EACH index (character and word) separately, in addition to the combined score. This will be useful in Step 3 as features.

In [7]:
# Make sure there are no missing normalized literals
test['Literal_norm'] = test['Literal_norm'].fillna('')

# Transform all test literals at once (this is fast)
test_char_vectors = char_vectorizer.transform(test['Literal_norm'])
test_word_vectors = word_vectorizer.transform(test['Literal_norm'])

print(f'Test character vectors: {test_char_vectors.shape}')
print(f'Test word vectors:      {test_word_vectors.shape}')

Test character vectors: (6667, 38809)
Test word vectors:      (6667, 29888)


In [8]:
print(f'Running retrieval on {len(test):,} test literals')
print('Expect this to take 3 to 6 minutes...')

BATCH = 100  # process this many literals at a time to keep memory usage low

retrieval_codes = []
retrieval_combined_scores = []
retrieval_char_scores = []
retrieval_word_scores = []
retrieval_descriptions = []

start = time.time()

for i in range(0, len(test), BATCH):
    end = min(i + BATCH, len(test))

    # Character similarities for this batch against all 180K references
    char_sims = cosine_similarity(test_char_vectors[i:end], char_ref_vectors)

    # Word similarities for this batch against all 180K references
    word_sims = cosine_similarity(test_word_vectors[i:end], word_ref_vectors)

    # Combine and pick the best match per literal
    combined_sims = (char_sims + word_sims) / 2
    best_idx = combined_sims.argmax(axis=1)

    # Store results for each literal in this batch
    for j in range(end - i):
        bi = best_idx[j]
        retrieval_codes.append(reference.iloc[bi]['Code'])
        retrieval_descriptions.append(reference.iloc[bi]['Description'])
        retrieval_combined_scores.append(float(combined_sims[j, bi]))
        retrieval_char_scores.append(float(char_sims[j, bi]))
        retrieval_word_scores.append(float(word_sims[j, bi]))

    # Free memory before next batch
    del char_sims, word_sims, combined_sims

    if (i // BATCH) % 10 == 0:
        elapsed = time.time() - start
        pct = end / len(test) * 100
        print(f'  {end:>5,} / {len(test):,} processed ({pct:.0f}%) in {elapsed:.1f}s')

# Attach the results to the test dataframe
test['retrieval_code']        = retrieval_codes
test['retrieval_desc']        = retrieval_descriptions
test['retrieval_score']       = retrieval_combined_scores
test['retrieval_char_score']  = retrieval_char_scores
test['retrieval_word_score']  = retrieval_word_scores

print(f'\nDone in {time.time()-start:.1f}s total')

Running retrieval on 6,667 test literals
Expect this to take 3 to 6 minutes...
    100 / 6,667 processed (1%) in 0.8s
  1,100 / 6,667 processed (16%) in 8.2s
  2,100 / 6,667 processed (31%) in 15.1s
  3,100 / 6,667 processed (46%) in 22.1s
  4,100 / 6,667 processed (61%) in 29.1s
  5,100 / 6,667 processed (76%) in 35.9s
  6,100 / 6,667 processed (91%) in 42.3s

Done in 46.0s total


## 6. Inspect the retrieval score distribution

In [9]:
print('Retrieval score statistics:')
print(test['retrieval_score'].describe())

print('\nScore buckets:')
for lo, hi in [(0, 0.1), (0.1, 0.2), (0.2, 0.3), (0.3, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 0.8), (0.8, 1.01)]:
    n = ((test.retrieval_score >= lo) & (test.retrieval_score < hi)).sum()
    print(f'  {lo:.2f} - {hi:.2f}: {n:5,} literals')

Retrieval score statistics:
count    6667.000000
mean        0.511896
std         0.257881
min         0.000000
25%         0.320147
50%         0.509576
75%         0.692465
max         1.000000
Name: retrieval_score, dtype: float64

Score buckets:
  0.00 - 0.10:   342 literals
  0.10 - 0.20:   550 literals
  0.20 - 0.30:   647 literals
  0.30 - 0.40:   788 literals
  0.40 - 0.50:   858 literals
  0.50 - 0.60: 1,062 literals
  0.60 - 0.80: 1,385 literals
  0.80 - 1.01: 1,035 literals


In [10]:
# Sample of the highest-confidence retrievals - these look very plausible
print('Highest-confidence retrievals:')
for _, r in test.nlargest(10, 'retrieval_score').iterrows():
    print(f'  [{r.retrieval_score:.3f}] "{r.Literal}" -> {r.retrieval_code}: {r.retrieval_desc}')

Highest-confidence retrievals:
  [1.000] "CARCINOMA LOBULAR IN SITU DE MAMA IZQUIERDA" -> D0502: Carcinoma lobular in situ de mama izquierda
  [1.000] "ENDOMETRIOSIS PROFUNDA DE OVARIO IZQUIERDO" -> N80122: Endometriosis profunda de ovario izquierdo
  [1.000] "GRUPO SANGUINEO B, Rh Positivo" -> Z6720: Grupo sanguíneo B, Rh positivo
  [1.000] "demencia vascular" -> F01: Demencia vascular
  [1.000] "Enfermedad de Méniere" -> H810: Enfermedad de Ménière
  [1.000] "demencia vascular" -> F01: Demencia vascular
  [1.000] "Colelitiasis" -> K80: Colelitiasis
  [1.000] "artrosis de rodilla" -> M17: Artrosis de rodilla
  [1.000] "NEOPLASIA MALIGNA DE PROSTATA" -> C61: Neoplasia maligna de próstata
  [1.000] "divertículo uretral" -> N361: Divertículo uretral


In [11]:
# Sample of low-confidence retrievals - these are probably wrong, but at least we have something
print('Lowest-confidence retrievals:')
for _, r in test.nsmallest(10, 'retrieval_score').iterrows():
    print(f'  [{r.retrieval_score:.3f}] "{r.Literal}" -> {r.retrieval_code}: {r.retrieval_desc}')

Lowest-confidence retrievals:
  [0.000] "VHC" -> A00: Cólera
  [0.000] "HPB" -> A00: Cólera
  [0.000] "cbp" -> A00: Cólera
  [0.000] "K43.2" -> A00: Cólera
  [0.000] "HPV" -> A00: Cólera
  [0.000] "RPBF" -> A00: Cólera
  [0.000] "652.81" -> A00: Cólera
  [0.000] "TGD" -> A00: Cólera
  [0.000] "npt" -> A00: Cólera
  [0.000] "RxTX" -> A00: Cólera


## 7. Validate Step 2 alone vs. Step 1 vs. Combined

Same train/val split we used in Step 1 (80/20, same random state) so the comparison is fair. We measure:

- Step 1 alone (exact + fuzzy from previous notebook logic)
- Step 2 alone (retrieval against the reference)
- Combined (Step 1 first, Step 2 as fallback)

In [12]:
# Same split as Step 1 to make comparisons fair
train['Literal_norm'] = train['Literal_norm'].fillna('')
train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
val_split = val_split.copy()

print(f'Validation set: {len(val_split):,} literals')

Validation set: 2,740 literals


In [13]:
# Compute retrieval predictions for the validation set
val_char = char_vectorizer.transform(val_split['Literal_norm'])
val_word = word_vectorizer.transform(val_split['Literal_norm'])

val_ret_codes = []
val_ret_scores = []

print('Running retrieval on validation set...')
start = time.time()

for i in range(0, len(val_split), BATCH):
    end = min(i + BATCH, len(val_split))
    cs = cosine_similarity(val_char[i:end], char_ref_vectors)
    ws = cosine_similarity(val_word[i:end], word_ref_vectors)
    combined = (cs + ws) / 2
    best_idx = combined.argmax(axis=1)
    for j in range(end - i):
        bi = best_idx[j]
        val_ret_codes.append(reference.iloc[bi]['Code'])
        val_ret_scores.append(float(combined[j, bi]))
    del cs, ws, combined

val_split['ret_code'] = val_ret_codes
val_split['ret_score'] = val_ret_scores
val_split['ret_correct'] = (val_split['ret_code'] == val_split['Code'])

print(f'Done in {time.time()-start:.1f}s')
print(f'Step 2 (retrieval) accuracy on validation: {val_split["ret_correct"].mean()*100:.1f}%')

Running retrieval on validation set...
Done in 17.9s
Step 2 (retrieval) accuracy on validation: 10.7%


In [14]:
# Reproduce Step 1 on the same split for a fair comparison
from rapidfuzz import fuzz, process

step1_lookup = {}
for lit_norm, group in train_split.groupby('Literal_norm'):
    step1_lookup[lit_norm] = group['Code'].value_counts().index[0]

val_split['s1_exact'] = val_split['Literal_norm'].map(step1_lookup)

# Fuzzy for the unmatched ones
val_unmatched = val_split[val_split['s1_exact'].isna()]
lookup_keys = list(step1_lookup.keys())

print(f'Running fuzzy matching on {len(val_unmatched):,} validation literals...')

fuzzy_codes = {}
for idx, row in val_unmatched.iterrows():
    r = process.extractOne(row['Literal_norm'], lookup_keys, scorer=fuzz.ratio, score_cutoff=60)
    if r is not None:
        fuzzy_codes[idx] = step1_lookup[r[0]]

# Combine exact + fuzzy into a single Step 1 prediction column
val_split['s1_code'] = val_split['s1_exact'].copy()
for idx, code in fuzzy_codes.items():
    val_split.loc[idx, 's1_code'] = code

val_split['s1_correct'] = (val_split['s1_code'] == val_split['Code'])

Running fuzzy matching on 1,266 validation literals...


In [15]:
# Combine: take Step 1 if available, otherwise Step 2
val_split['combined_code'] = val_split['s1_code']
missing = val_split['combined_code'].isna()
val_split.loc[missing, 'combined_code'] = val_split.loc[missing, 'ret_code']
val_split['combined_correct'] = (val_split['combined_code'] == val_split['Code'])

In [16]:
# Summary table
n = len(val_split)
s1_overall = val_split['s1_correct'].sum() / n * 100
s2_overall = val_split['ret_correct'].sum() / n * 100
cb_overall = val_split['combined_correct'].sum() / n * 100

print('Validation comparison')
print(f'{"":35s} {"correct":>10s} {"of total":>10s}')
print('-' * 60)
print(f'{"Step 1 only (exact + fuzzy)":35s} {val_split.s1_correct.sum():>10,} {s1_overall:>9.1f}%')
print(f'{"Step 2 only (retrieval)":35s} {val_split.ret_correct.sum():>10,} {s2_overall:>9.1f}%')
print(f'{"Step 1 + Step 2 (combined)":35s} {val_split.combined_correct.sum():>10,} {cb_overall:>9.1f}%')

Validation comparison
                                       correct   of total
------------------------------------------------------------
Step 1 only (exact + fuzzy)              1,040      38.0%
Step 2 only (retrieval)                    294      10.7%
Step 1 + Step 2 (combined)               1,043      38.1%


In [17]:
# Where does each method win and lose? This shows how complementary they are.
s1f = val_split['s1_correct'].fillna(False)
s2f = val_split['ret_correct'].fillna(False)

both_right = (s1f & s2f).sum()
only_s1 = (s1f & ~s2f).sum()
only_s2 = (~s1f & s2f).sum()
both_wrong = (~s1f & ~s2f).sum()

print('Overlap between Step 1 and Step 2 predictions:')
print(f'  Both correct:     {both_right:>5,} ({both_right/n*100:.1f}%)')
print(f'  Only Step 1 right: {only_s1:>5,} ({only_s1/n*100:.1f}%)')
print(f'  Only Step 2 right: {only_s2:>5,} ({only_s2/n*100:.1f}%)')
print(f'  Both wrong:       {both_wrong:>5,} ({both_wrong/n*100:.1f}%)')
print()
print('The "only Step 2 right" cases are exactly the literals where retrieval')
print('adds value beyond matching - these are the cases Step 1 missed.')

Overlap between Step 1 and Step 2 predictions:
  Both correct:       157 (5.7%)
  Only Step 1 right:   883 (32.2%)
  Only Step 2 right:   137 (5.0%)
  Both wrong:       1,563 (57.0%)

The "only Step 2 right" cases are exactly the literals where retrieval
adds value beyond matching - these are the cases Step 1 missed.


## 8. Build the final Step 2 predictions on the real test set

For each test literal:
- If Step 1 produced a prediction (exact or fuzzy), keep it
- Otherwise, use the retrieval prediction from Step 2

We also keep all the retrieval scores in the output, because Step 3 will use them as features.

In [18]:
# Merge Step 1 predictions into the test dataframe so everything is in one place
test = test.merge(step1[['id', 'step1_code', 'step1_method']], on='id', how='left')

# Initialize final columns from Step 1
test['final_code'] = test['step1_code']
test['final_method'] = test['step1_method']

# Where Step 1 was unresolved, fall back to retrieval
needs_fallback = test['final_code'].isna()
test.loc[needs_fallback, 'final_code'] = test.loc[needs_fallback, 'retrieval_code']
test.loc[needs_fallback, 'final_method'] = 'retrieval'

print('Final method breakdown:')
print(test['final_method'].value_counts())
print()
print(f'Total predictions: {test["final_code"].notna().sum():,} / {len(test):,}')

Final method breakdown:
final_method
exact        3812
fuzzy        2819
retrieval      36
Name: count, dtype: int64

Total predictions: 6,667 / 6,667


In [ ]:
# Save everything for Step 3
output_cols = [
    'id', 'Literal', 'Literal_clean', 'Literal_norm',
    'step1_code', 'step1_method',
    'retrieval_code', 'retrieval_desc',
    'retrieval_score', 'retrieval_char_score', 'retrieval_word_score',
    'final_code', 'final_method'
]
output = test[output_cols].copy()
output.to_csv('step2_predictions.csv', index=False)

print('Saved step2_predictions.csv')
print(f'\nSample rows:')
output[['Literal', 'final_method', 'final_code', 'retrieval_score']].sample(10, random_state=7)

Saved step2_predictions.csv

Sample rows:


,Literal,final_method,final_code,retrieval_score
6592,SEPSIS rn,exact,P369,0.751662
1698,pródromos,fuzzy,O624,0.169195
2930,ECO TV,exact,8878,0.151156
5818,Febrícula,exact,R509,0.214536
2109,desgarro grado 2º,exact,0KQM0ZZ,0.529389
3430,extracción manual placentaria,fuzzy,10D17Z9,0.330496
3580,E. Coli,exact,04149,0.457790
6441,feto único en podálica convertida anteparto,fuzzy,64623,0.327310
2699,Polipectomía endometrial,exact,0UB98ZZ,0.635359
5988,Quistes hepáticos,exact,K7689,0.411516


: 